In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import os

: 

In [ ]:
import torch
import numpy as np
import pandas as pd

import os
from tqdm import tqdm
import random

from functools import partial
import TransformerLens.transformer_lens.utils as utils
from TransformerLens.transformer_lens import patching
from jaxtyping import Float

import plotly.express as px
import plotly.io as pio

from helpers import (
    load_json_file,
    load_tokenizer_and_models,
)

from patching_helpers import (
    get_activations,
    get_act_patch_attn_head_out_all_pos,
    patch_specific_head_attn_z_all_pos,
    patch_specific_multi_heads_attn_z_all_pos
)
import numpy as np
import csv
from tqdm import tqdm
from sklearn.linear_model import LinearRegression
from itertools import product
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
import numpy as np
from sklearn.inspection import permutation_importance
import math
def BM_score(tokenized_pair_original, doc_token_len, L, k=1.5, b=0.75, idf_matrix_row=None):
    """
    Calculate the BM25 score between a query and a document.

    Parameters:
    - tokenized_pair_original (dict): A dictionary with the "input_ids" for query and document.
    - doc_token_len (int): The length of the document in tokens.
    - L (float): The average document length in the corpus.
    - k (float): Term frequency saturation parameter (default 1.5).
    - b (float): Length normalization parameter (default 0.75).
    - idf_matrix_row (dict or list): Structure to get the IDF for each term (indexed by query_id).
    
    Returns:
    - float: BM25 score
    """
    # Extract query and document tokens
    query_ids = tokenized_pair_original["input_ids"][0][1:6].tolist()
    doc_ids = tokenized_pair_original["input_ids"][0][7:-1].tolist() # [6+1:-1] adjusted for simpler indexing
    
    # Calculate BM25 score
    bm25_score = 0.0
    for query_id in query_ids:
        # Directly count occurrences of the query term in the document
        tf = doc_ids.count(query_id)
        #print(f"Term Frequency (TF) for token {query_id}: {tf}")
        
        if tf == 0:
            continue  # Skip terms that don't appear in the document
        
        # Retrieve IDF for the current query term
        #idf = idf_matrix_row[query_id] if idf_matrix_row else math.log((1 + len(doc_ids)) / (1 + tf)) + 1
        idf = idf_matrix_row[query_id] if idf_matrix_row is not None and len(idf_matrix_row) > 0 else math.log((1 + len(doc_ids)) / (1 + tf)) + 1

        
        # BM25 term score calculation
        term_score = idf * (tf * (k + 1)) / (tf + k * (1 - b + b * (doc_token_len / L)))
        bm25_score += term_score

    return bm25_score

pre_trained_model_name = "cross-encoder/ms-marco-MiniLM-L-12-v2"
torch.set_grad_enabled(False)
device = utils.get_device()
tokenizer, tl_model, pooler_layer,dropout_layer,classifier_layer = load_tokenizer_and_models(pre_trained_model_name, device)
selected_query_terms = {"1089763": "miners", "1089401": "tsca", "1088958": "cadi", "1088541": "fletcher,nc", "1088475": "holmes,ny", "1101090": "azadpour", "1088444": "kashan", "1085779": "canopius", "1085510": "carewell", "1085348": "polson", "1085229": "wendelville", "1100499": "trematodiases", "1100403": "arcadis", "1064808": "acantholysis", "1100357": "ardmore", "1062223": "animsition", "1058515": "cladribine", "1051372": "cineplex", "1048917": "misconfiguration", "1045135": "wellesley", "1029552": "tosca", "1028752": "watamote", "1099761": "ari", "1020376": "amplicons", "1002940": "iheartradio", "1000798": "alpha", "992257": "desperation", "197024": "greenhorns", "61277": "brat", "44072": "chatsworth", "195582": "dammam", "234165": "saluki", "196111": "gorm", "329958": "pesto", "100020": "cortana", "193866": "izzam", "448976": "potsherd", "575616": "ankole", "434835": "konig", "488676": "retinue", "389258": "hughes", "443081": "lotte", "511367": "nfcu", "212477": "ouachita", "544060": "dresden", "428773": "wunderlist", "478295": "tigard", "610132": "neodesha", "435412": "lakegirl", "444350": "mageirocophobia", "492988": "saptco", "428819": "swegway", "477286": "antigonish", "478054": "paducah", "1094996": "tacko", "452572": "mems", "20432": "aqsarniit", "559709": "plectrums", "748935": "fraenulum?", "482666": "defdinition", "409071": "ecpi", "1101668": "denora", "537995": "cottafavi", "639084": "hortensia", "82161": "windirstat", "605651": "emmett", "720013": "arzoo", "525047": "trumbull", "978802": "browerville", "787784": "provocative", "780336": "orthorexia", "1093438": "lickspittle", "788851": "qualfon", "61531": "campagnolo", "992652": "setaf", "1092394": "msdcf", "860942": "viastone", "863187": "wintv", "1092159": "northwoods", "990010": "paihia", "840445": "prentice-hall", "775355": "natamycin", "986325": "lapham", "1091654": "parisian", "768411": "mapanything?", "194724": "gesundheit", "985905": "sentral", "1091206": "putrescine", "760930": "islet", "1090945": "ryder", "1090839": "bossov", "1090808": "semispinalis", "774866": "myfortic", "820027": "lithotrophy", "798967": "spredfast", "126821": "scooped", "60339": "stroganoff", "1090374": "strategery", "180887": "enu", "292225": "molasses"}
fbase_path = ""
tfc1_add_queries = pd.read_csv(os.path.join(fbase_path, "tfc1_add_qids_with_text.csv"), header=None, names=["_id", "text"])
tfc1_queries_dict = tfc1_add_queries.set_index('_id').to_dict(orient='index')
tfc1_add_baseline_corpus = load_json_file(os.path.join(fbase_path, "tfc1_add_baseline_final_dd_append_corpus.json"))["corpus"]
target_qids = tfc1_add_queries["_id"].tolist() #[448976] 
tfc1_add_queries = tfc1_add_queries[tfc1_add_queries["_id"].isin(target_qids)] #tfc remains unchanged
tfc1_add_dd_corpus = load_json_file(os.path.join(fbase_path, f"tfc1_add_append_final_dd_corpus.json"))["corpus"]

import torch
import numpy as np
from tqdm import tqdm
idf_file_path = 'msmarco_idf.tsv'
# Load the IDF data into a DataFrame
idf_df = pd.read_csv(idf_file_path, sep='\t', header=None, names=['word', 'idf'])
# Convert the DataFrame to a dictionary for quick lookup
idf_dict = pd.Series(idf_df.idf.values, index=idf_df.word).to_dict()
idf_list = []

for i in range(tl_model.cfg.d_vocab):
    token =tokenizer.decode(i)

    #print(token)
    idf = idf_dict.get(token)
    if idf!=None:
        idf_list.append(idf)
        
    else:
        idf_list.append(0)

idf_matrix = torch.tensor(idf_list)

print(idf_matrix.shape)
idf_matrix_row = idf_matrix.squeeze().to(tl_model.W_E.device)

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



# Function to perform linear regression and output metrics
def linear_reg_tests(features, model_scores):
    # Fit linear regression model on the entire dataset
    model = LinearRegression()
    model.fit(features, model_scores)
    print(model_scores.shape)
    # Predict on the same dataset
    y_pred = model.predict(features)

    # Calculate metrics
    test_corr, _ = pearsonr(model_scores, y_pred)
    rmse = np.sqrt(mean_squared_error(model_scores, y_pred))
    r2 = r2_score(model_scores, y_pred)
    rank_corr, _ = spearmanr(model_scores, y_pred)

    # Print metrics
    print("Just fitting the data (no train test split)")
    print(f"Pearson Correlation: {test_corr:.4f}")
    print(f"Spearman Rank Correlation: {rank_corr:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R² Score: {r2:.4f}")
    return model
def linear_reg_train_test(features,model_scores):
    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, model_scores, test_size=0.2, random_state=42
    )

    # Fit linear regression model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict on test set
    y_pred = model.predict(X_test)

    # Calculate metrics
    test_corr, p_pearson = pearsonr(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    rank_corr, p_spearman = spearmanr(y_test, y_pred)
        # Print metrics
    print("Train Test split")
    print(f"Pearson Correlation: {test_corr:.4f}, p value{p_pearson:.4f}")
    print(f"Spearman Rank Correlation: {rank_corr:.4f},p value {p_spearman:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R² Score: {r2:.4f}")
    return model

    # Print metrics

mps.is_available()


/Users/meng.lu/miniforge3/envs/tensor_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
If using BERT for interpretability research, keep in mind that BERT has some significant architectural differences to GPT. For example, LayerNorms are applied *after* the attention and MLP components, meaning that the last LayerNorm in a block cannot be folded.


will start convert_hf_model_config
Moving model to device:  mps
Loaded pretrained model cross-encoder/ms-marco-MiniLM-L-12-v2 into HookedEncoder


ImportError: dlopen(/Users/meng.lu/miniforge3/envs/tensor_env/lib/python3.10/site-packages/tensorflow/python/platform/_pywrap_cpu_feature_guard.so, 0x0002): Library not loaded: @rpath/libtensorflow_cc.2.dylib
  Referenced from: <DEACBD47-7DD9-3FF5-B083-FDBB52F55ABD> /Users/meng.lu/miniforge3/envs/tensor_env/lib/python3.10/site-packages/tensorflow/python/_pywrap_tensorflow_internal.so
  Reason: tried: '/Users/meng.lu/miniforge3/envs/tensor_env/lib/python3.10/site-packages/tensorflow/python/../../_solib_darwin_arm64/libtensorflow_cc.2.dylib' (no such file), '/Users/meng.lu/miniforge3/envs/tensor_env/lib/python3.10/site-packages/tensorflow/python/../../_solib_darwin_arm64/_U_S_Stensorflow_Spython_C_Upywrap_Utensorflow_Uinternal.so_Ucclib___Utensorflow/libtensorflow_cc.2.dylib' (no such file), '/Users/meng.lu/miniforge3/envs/tensor_env/lib/python3.10/site-packages/tensorflow/python/libtensorflow_cc.2.dylib' (no such file), '/Users/meng.lu/miniforge3/envs/tensor_env/lib/python3.10/site-pack

In [ ]:


W_E = tl_model.W_E
U, S, Vt = torch.linalg.svd(W_E, full_matrices=False)
U_0 = U[:, 0]


#I want to align U_0 values with the distribution of idf_matrix_row, same mean same standard deviation
# Compute mean and standard deviation of idf_matrix_row
idf_mean = idf_matrix_row.mean()
idf_std = idf_matrix_row.std()

# Compute mean and standard deviation of U_0
U_0_mean = U_0.mean()
U_0_std = U_0.std()

# Apply linear transformation to U_0 to align with idf_matrix_row
U0_idf = (U_0 - U_0_mean) / U_0_std * idf_std + idf_mean

# Check the result
print(f"U_0 aligned mean: {U0_idf.mean().item()}, std: {U0_idf.std().item()}")
print(f"IDF mean: {idf_mean.item()}, IDF std: {idf_std.item()}")

def linear_approximation(ALL_MATCHING_HEADS,idf_matrix_row):
   
    features = []
    model_scores = []
    bm25_scores = []

    # Iterate through each query ID (QID)
    for i, qid in enumerate(tqdm(target_qids)):
        query = tfc1_queries_dict[qid]['text']
        selected_query_token = selected_query_terms[str(qid)]
        target_docs = tfc1_add_dd_corpus[str(qid)]
        
        for j, doc_id in enumerate(target_docs):
            if j < 1: 
                perturbed_doc = tfc1_add_dd_corpus[str(qid)][doc_id]["text"]
                tokenized_pair_original = tokenizer([query], [perturbed_doc], return_tensors="pt", padding=True, truncation=True)
                input_list = tokenized_pair_original['input_ids'][0].tolist()
                fir_SEP_position = input_list.index(102)
                doc_token_len = len(input_list[fir_SEP_position + 1:-1])
                query_len = fir_SEP_position - 1

                
                if query_len >= 6:
                    


                    concatenated_input_ids = torch.cat((
                        tokenized_pair_original["input_ids"][0][:6], tokenized_pair_original["input_ids"][0][fir_SEP_position:]
                    ))
                    tokenized_pair_original["input_ids"] = torch.unsqueeze(concatenated_input_ids, 0)
                    concatenated_attn_mask = torch.cat((
                        tokenized_pair_original["attention_mask"][0][:6], tokenized_pair_original["attention_mask"][0][fir_SEP_position:]
                    ))
                    tokenized_pair_original["attention_mask"] = torch.unsqueeze(concatenated_attn_mask, 0)
                    concatenated_token_type_ids = torch.cat((
                        tokenized_pair_original["token_type_ids"][0][:6], tokenized_pair_original["token_type_ids"][0][fir_SEP_position:]
                    ))
                    tokenized_pair_original["token_type_ids"] = torch.unsqueeze(concatenated_token_type_ids, 0)

                    # Prepare activation names based on ALL_MATCHING_HEADS
                    names_list = [utils.get_act_name('pattern', layer) for layer, head in ALL_MATCHING_HEADS]
                    orig_outputs, act_orig = tl_model.run_with_cache(
                        tokenized_pair_original["input_ids"],
                        return_type="embeddings",
                        one_zero_attention_mask=tokenized_pair_original["attention_mask"],
                        token_type_ids=tokenized_pair_original['token_type_ids'],
                        names_filter=lambda name: name in names_list,
                    )

                    y = model_score = classifier_layer(dropout_layer(pooler_layer(orig_outputs))).item()
                    feature_list = []

                    # Loop over each head to extract corresponding features
                    

                    for tok_index in range(1, 6):
                        

                        ft1 = idf_tok = -U0_idf[input_list[tok_index]].item()

                        feature_list.append(ft1)
                        for layer, head in ALL_MATCHING_HEADS:
                            pattern_name = utils.get_act_name('pattern', layer)
                            pattern_orig = act_orig[pattern_name][0, head, :, :].cpu().numpy()
                            other_tok_indices = [t for t in range(fir_SEP_position + 1, len(tokenized_pair_original["input_ids"][0])) if t != tok_index]
                            attention_scores_on_other_tokens = pattern_orig[tok_index, other_tok_indices].sum()
                            ft2 = attention_scores_on_other_tokens
                            ft3 = ft1 * ft2
                            feature_list.extend([ft2,ft3])

                    # Append features and model score for each document
                    #print(len(feature_list))
                    features.append(feature_list)
                    model_scores.append(y)

                    
                    BM_score_result = BM_score(tokenized_pair_original,doc_token_len,L=84.32,k=2,b=0.75,idf_matrix_row=idf_matrix_row).cpu()
                    bm25_scores.append(BM_score_result)

    features = np.array(features)
    model_scores = np.array(model_scores)
    means = features.mean(axis=0)
    stds = features.std(axis=0)

    stds[stds == 0] = 1
    def is_valid(entry):
        return entry is not None and isinstance(entry, (int, float, np.ndarray))

    # Filter valid entries
    valid_indices = [
        i for i, (f, ms, bm) in enumerate(zip(features, model_scores, bm25_scores))
        if is_valid(f) and is_valid(ms) and is_valid(bm)
    ]
    # Filter the arrays
    features = features[valid_indices]
    model_scores = model_scores[valid_indices]
    bm25_scores = bm25_scores[valid_indices]
    model_bm25_corr, _ = pearsonr(model_scores, bm25_scores)
    print(f"Pearson Correlation between model_scores and bm25_scores: {model_bm25_corr:.4f}")
    # Perform linear regression on the actual features
    print("Results with actual features:")
    model = linear_reg_tests(features, model_scores)
    model = linear_reg_train_test(features, model_scores)
    print("bm 25")
    linear_reg_train_test(features, bm25_scores)
    random_features = np.random.normal(features.mean(axis=0), features.std(axis=0), features.shape)
    # Perform linear regression on the randomly generated features
    print("\n\nResults with randomly generated features:")
    linear_reg_tests(random_features, model_scores)
    print("bm 25")
    linear_reg_train_test(random_features, model_scores)
    linear_reg_train_test(random_features, bm25_scores)

mps.is_available()


If using BERT for interpretability research, keep in mind that BERT has some significant architectural differences to GPT. For example, LayerNorms are applied *after* the attention and MLP components, meaning that the last LayerNorm in a block cannot be folded.


will start convert_hf_model_config
Moving model to device:  mps
Loaded pretrained model cross-encoder/ms-marco-MiniLM-L-12-v2 into HookedEncoder
torch.Size([30522])
U_0 aligned mean: 5.253446102142334, std: 4.0815300941467285
IDF mean: 5.253446102142334, IDF std: 4.081530570983887


100%|██████████| 100/100 [00:32<00:00,  3.05it/s]



Linear Reg Pearson Correlation: 0.9999999999999999
Random Forest Pearson Correlation: 0.5254640365146428
feature importance [1.40026164e-02 2.28657942e-03 2.84531794e-03 3.07045245e-04
 3.72810880e-03 9.61968410e-03 1.28089109e-03 1.18579992e-02
 8.52332228e-04 2.25110719e-03 4.37545317e-03 2.84052848e-02
 1.88098403e-02 6.72598211e-05 1.21067381e-03 9.13547927e-04
 3.82570787e-04 1.17346481e-03 1.76224981e-03 1.55327152e-03
 1.84667448e-03 9.22276590e-03 1.81886076e-03 1.27483103e-03
 6.93374409e-02 2.57333096e-02 3.89869752e-03 2.01958201e-03
 7.13840328e-03 1.26735189e-04 1.29362937e-03 5.43432758e-04
 3.38478740e-02 9.18108604e-03 1.65187277e-03 2.01719290e-03
 1.36062686e-02 7.25145536e-03 8.15939899e-03 1.93971523e-04
 1.24282452e-03 3.91687992e-03 8.57353698e-04 9.78604676e-05
 7.99803796e-03 7.70021633e-03 1.18397223e-04 2.73737721e-03
 5.87250299e-03 8.48588424e-03 1.91628860e-03 6.51989936e-03
 3.33049902e-03 8.38589041e-03 6.72609258e-04 2.16343728e-03
 3.81197343e-03 1.049

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# Load the original features and model scores
features = np.load('MatchingHeads/msmarco/features_msmarco.npy', allow_pickle=True)
model_scores = np.load('MatchingHeads/msmarco/model_scores_msmarco.npy', allow_pickle=True)
bm25_scores = np.load('MatchingHeads/msmarco/BM_scores_msmarco.npy', allow_pickle=True)


# Function to check for None or invalid entries
def is_valid(entry):
    return entry is not None and isinstance(entry, (int, float, np.ndarray))

# Filter valid entries
valid_indices = [
    i for i, (f, ms, bm) in enumerate(zip(features, model_scores, bm25_scores))
    if is_valid(f) and is_valid(ms) and is_valid(bm)
]
# Filter the arrays
features = features[valid_indices]
model_scores = model_scores[valid_indices]
bm25_scores = bm25_scores[valid_indices]
model_bm25_corr, _ = pearsonr(model_scores, bm25_scores)
print(f"Pearson Correlation between model_scores and bm25_scores: {model_bm25_corr:.4f}")

# Perform linear regression on the actual features
print("Results with actual features:")
model = linear_reg_tests(features, model_scores)
model = linear_reg_train_test(features, model_scores)
print("bm 25")
linear_reg_train_test(features, bm25_scores)
#add: test the perason correaltion between model_scores and bm25_scores



# Generate random features with the same distribution as the original features
random_features = np.random.normal(features.mean(axis=0), features.std(axis=0), features.shape)

# Perform linear regression on the randomly generated features
print("\n\nResults with randomly generated features:")
linear_reg_tests(random_features, model_scores)
print("bm 25")
linear_reg_train_test(random_features, model_scores)

Pearson Correlation between model_scores and bm25_scores: 0.1595
Results with actual features:
(7600,)
Just fitting the data (no train test split)
Pearson Correlation: 0.8351
Spearman Rank Correlation: 0.8152
RMSE: 2.4533
R² Score: 0.6974
Train Test split
Pearson Correlation: 0.8157
Spearman Rank Correlation: 0.8010
RMSE: 2.5542
R² Score: 0.6649
bm 25
Train Test split
Pearson Correlation: 0.6939
Spearman Rank Correlation: 0.6864
RMSE: 5.5449
R² Score: 0.4788


Results with randomly generated features:
(7600,)
Just fitting the data (no train test split)
Pearson Correlation: 0.1409
Spearman Rank Correlation: 0.1326
RMSE: 4.4156
R² Score: 0.0198
bm 25
Train Test split
Pearson Correlation: 0.0397
Spearman Rank Correlation: 0.0393
RMSE: 4.4378
R² Score: -0.0115


LinearRegression()